# **Package & Imports**

`package org.tiaa.online.pages;`


This class is inside org.tiaa.online.pages.

Organizes code so it doesn’t clash with other classes (like DriverManager).
`#%% md
`

`import org.openqa.selenium.By;
import org.openqa.selenium.JavascriptExecutor;
import org.openqa.selenium.WebDriver;
import org.openqa.selenium.WebElement;
import org.openqa.selenium.support.ui.ExpectedConditions;
import org.openqa.selenium.support.ui.WebDriverWait;
import utils.com.transflower.learning.DriverManager;
import java.time.Duration;
`

👉Each import has a purpose:
By → used to locate elements (id, xpath, css, etc.).

JavascriptExecutor → allows execution of JS (e.g., check document state).

WebDriver → main interface to control browser.

WebElement → represents elements inside the page (like input fields, buttons).

ExpectedConditions → common wait conditions (visible, clickable).

WebDriverWait → explicit wait utility.

DriverManager → your custom class that manages WebDriver lifecycle.

Duration → Java’s modern time API, replaces old int seconds.

TestData → custom class for validating or storing test data (used for URL validation).


# Class Declaration
`
public class BasePage {`


public: accessible everywhere.

BasePage: foundation class for all Page Objects (LoginPage, HomePage, etc.).

Follows Page Object Model (POM) → reduces duplication, centralizes code.


# **Instance Variables**

    protected WebDriver driver;
    protected WebDriverWait wait;


driver: WebDriver instance. Marked protected so child classes (LoginPage) can use it.

wait: WebDriverWait instance (for explicit waits).


# **Constructor**

    public BasePage(){
        this.driver = DriverManager.getDriver();
        this.wait = new WebDriverWait(driver, Duration.ofSeconds(10));
    }


Constructor runs when any page object is created.

DriverManager.getDriver() fetches the driver created in DriverManager.

new WebDriverWait(driver, Duration.ofSeconds(10)) → sets up an explicit wait with 10s timeout.

⚡ This ensures every page object automatically has driver + wait initialized.

# **Navigate to URL**

    public void naviagteTo(String url) {
        System.out.println("navigating to" + url);
        driver.get(url);


Logs URL in console.

Opens given URL in browser.



        String currentUrl = driver.getCurrentUrl();
        System.out.println("current url after navigation" + currentUrl);
Fetches the actual current URL.

Logs it → useful for debugging (sometimes redirects happen).


        TestData.URLs.ValidateURL(currentUrl, url);
Calls a custom validation function.

Ensures the actual URL matches the expected one.

Adds a soft assertion mechanism inside BasePage itself.


        waitForPageLoad();
    }
Waits until the page is completely loaded before continuing.



# **Wait for Page Load**

    private void waitForPageLoad() {
        wait.until(webDriver ->
            ((JavascriptExecutor) webDriver).executeScript("return document.readyState")
                .equals("complete"));
    }


Executes JavaScript: document.readyState.

Possible states: "loading", "interactive", "complete".

Waits until "complete" → meaning DOM + resources are ready.

⚡ This prevents flaky tests where Selenium tries to interact before the page is ready.

# **Click Element**

    protected void clickElement(By locator) {
        WebElement element = wait.until(ExpectedConditions.elementToBeClickable(locator));
    }

Finds element after waiting until it’s clickable.
Add safeClick() for stubborn elements that don’t click normally:

`protected void safeClick(By locator){
    try {
        clickElement(locator);
    } catch (Exception e) {
        WebElement element = driver.findElement(locator);
        ((JavascriptExecutor) driver).executeScript("arguments[0].click();", element);
    }
}
`

# **Send Keys (Type into Input)**

    protected void sendKeys(By locator, String text) {
        WebElement element = wait.until(ExpectedConditions.visibilityOfElementLocated(locator));
        element.clear();
        element.sendKeys(text);
    }


Waits for element to be visible.

Clears any existing text (element.clear()).

Types new text (element.sendKeys(text)).

⚡ Useful for filling forms.

# **Get Text**

    protected String getText(By locator) {
        WebElement element = wait.until(ExpectedConditions.visibilityOfElementLocated(locator));
        return element.getText();
    }


Waits until visible, then gets element text.

Commonly used for assertions (assertEquals(getText(locator), "Expected")).

# **Check Element Presence**

    protected boolean isElementPresent(By locator) {
        try {
            wait.until(ExpectedConditions.visibilityOfElementLocated(locator));
            return true;
        } catch (Exception ex) {
            return false;
        }
    }


Tries to find element with wait.

If found → true.

If not found (timeout/exception) → false.

⚡ Safer than driver.findElement() because it won’t break test immediately

# **Check Element Displayed**

    protected boolean isElementDisplayed(By locator) {
        try {
            return driver.findElement(locator).isDisplayed();
        } catch (Exception ex) {
            return false;
        }
    }


Directly calls isDisplayed().

If element not found → returns false.

⚡ Good for conditional checks.

# **Explicit Waits (Helpers)**

    protected void waitForElementToBeVisible(By locator) {
        wait.until(ExpectedConditions.visibilityOfElementLocated(locator));
    }

    protected void waitForElementTobeClickable(By locator) {
        wait.until(ExpectedConditions.elementToBeClickable(locator));
    }


Utility methods for waits.

Prevents writing wait.until() everywhere in test code.

# **Get Current Page Details**

    public String getCurrentUrl() {
        return driver.getCurrentUrl();
    }

    public String getPageTitle() {
        return driver.getTitle();
    }
}


Returns current URL (for validation).

Returns page title (useful for assertions like "Login Page").

In [ ]:
⚡ Why This BasePage is Powerful

Centralizes all common Selenium actions.

Handles waits → removes flakiness.

Provides reusable methods (click, sendKeys, getText).

Ensures page load is complete before interacting.

Serves as a parent class for all page objects → DRY principle (don’t repeat yourself).